In [2]:
import pandas as pd
df_imaging=pd.read_csv('cleaned_radio.csv')
df_labs=pd.read_csv('cleaned_lab.csv')
df_truth=pd.read_csv('cleaned_truth.csv')
df_discharge=pd.read_csv('cleaned_discharge_summary.csv')
df_outpatient=pd.read_csv('cleaned_outpatient_summary.csv')
df_endoscope=pd.read_csv('cleaned_endoscope.csv')

C:\Users\sdtc\AppData\Local\Temp\ipykernel_17632\447803074.py:3: DtypeWarning: Columns (1,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_labs=pd.read_csv('cleaned_lab.csv')


### Merge Radio and Clinical Results
if either `event` evaluated from radiology report is 1 or `event` evaluated from clinical reports (discharge and outpatient) is 1, then final output is 1\
usage below is for Ascites

In [3]:
import pandas as pd

# =====================================
# 1. STORE ALL YOUR MAPPINGS HERE
# =====================================
MAPPINGS = {
    "ascites": {
        "id": "Random ID",
        "radio_presence": "Radio_Ascites_presence",
        "radio_date": "Radio_Ascites_date",
        "clinical_presence": "Clinical_Ascites_presence",
        "clinical_date": "Clinical_Ascites_date",
        "output_prefix": "Ascites"
    },
    "HCC": {
        "id": "Random ID",
        "radio_presence": "Radio_HCC_presence",
        "radio_date": "Radio_HCC_date",
        "clinical_presence": "HCC_presence",
        "clinical_date": "HCC_date",
        "output_prefix": "HCC"
    }
}


# =====================================
# 2. MAIN FUNCTION (RUNS ONE MERGE ONLY)
# =====================================
def combine_presence_files(radio_file, clinical_file, mapping, output_csv):

    # Load files
    rdf = pd.read_csv(radio_file)
    cdf = pd.read_csv(clinical_file)

    id_col = mapping["id"]
    prefix = mapping["output_prefix"]

    # Clean ID
    for df in [rdf, cdf]:
        df[id_col] = (
            pd.to_numeric(df[id_col], errors='coerce')
            .fillna(0)
            .astype(int)
        )

    # Rename + keep needed cols
    rdf = rdf.rename(columns={
        mapping["radio_presence"]: "radio_presence",
        mapping["radio_date"]: "radio_date"
    })[[id_col, "radio_presence", "radio_date"]]

    cdf = cdf.rename(columns={
        mapping["clinical_presence"]: "clinical_presence",
        mapping["clinical_date"]: "clinical_date"
    })[[id_col, "clinical_presence", "clinical_date"]]

    # Merge
    merged = pd.merge(rdf, cdf, on=id_col, how="outer")

    # Normalize presence
    for col in ["radio_presence", "clinical_presence"]:
        merged[col] = (
            pd.to_numeric(merged[col], errors="coerce")
            .fillna(0)
            .astype(int)
        )

    # Any presence
    merged[f"Any_{prefix}_presence"] = (
        (merged["radio_presence"] == 1) |
        (merged["clinical_presence"] == 1)
    ).astype(int)

    # Earliest date
    merged[f"Any_{prefix}_date"] = pd.to_datetime(
        merged[["radio_date", "clinical_date"]]
        .apply(pd.to_datetime, errors="coerce")
        .min(axis=1)
    )

    # Final output
    final_df = merged[[id_col,
                       f"Any_{prefix}_presence",
                       f"Any_{prefix}_date"]]

    final_df.to_csv(output_csv, index=False)

    print(f"✅ Saved: {output_csv}")
    print(f"✅ Rows: {len(final_df)}")

    return final_df




### Export Merged Ascites Variable
`"Any_Ascites_presence"`

In [ ]:

# =====================================
# 3. RUN (CHANGE ONLY THIS PART)
# =====================================
if __name__ == "__main__":

    condition = "ascites"   # 👈 CHANGE THIS (ascites / edema / tumor)

    combine_presence_files(
        radio_file="1k RASCITES 6.5.csv",
        clinical_file="1k CASCITES 6.5.csv",
        mapping=MAPPINGS["ascites"],
        output_csv="1k AASCITES.csv"
    )

### Export Merged HCC Variable
`"Any_HCC_presence"`

In [ ]:
if __name__ == "__main__":
    condition =  "HCC"  # 👈 CHANGE THIS (ascites / edema / tumor)

    combine_presence_files(
        radio_file="1k RHCC 20.5.csv",
        clinical_file="1K HCC after_adjustments 15.5.csv",
        mapping=MAPPINGS["HCC"],
        output_csv="1k ANY_HCC.csv"
    )
    

### Binary Classification Evaluation
accuracy, precision, recall, and f1-score

In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from typing import Union, List


def calculate_medical_metrics_with_dates(
    extracted_csv_paths: Union[str, List[str]]
):
    """
    Assumes:
    - df_truth is already loaded
    - Not evaluated == 0 (by design of LLM pipeline)
    """

    # =========================
    # 1. LOAD EXTRACTED CSVs
    # =========================
    if isinstance(extracted_csv_paths, str):
        extracted_csv_paths = [extracted_csv_paths]

    extracted_dfs = [pd.read_csv(p) for p in extracted_csv_paths]
    final_extracted_df = pd.concat(extracted_dfs, ignore_index=True)

    # =========================
    # 2. CLEAN RANDOM ID
    # =========================
    for df in [df_truth, final_extracted_df]:
        df['Random ID'] = (
            pd.to_numeric(df['Random ID'], errors='coerce')
            .fillna(0)
            .astype(int)
        )

    # =========================
    # 3. NORMALIZE PRESENCE COLUMNS
    # =========================
    for col in final_extracted_df.columns:
        if col.endswith('_presence'):
            final_extracted_df[col] = (
                pd.to_numeric(final_extracted_df[col], errors='coerce')
                .fillna(0)
                .astype(int)
            )

    # =========================
    # 4. AGGREGATE (PATIENT LEVEL)
    # =========================
    agg_rules = {}
    for col in final_extracted_df.columns:
        if col == 'Random ID':
            continue
        if col.endswith('_presence'):
            agg_rules[col] = 'max'   # ANY positive survives
        else:
            agg_rules[col] = 'first'

    final_merged_df = (
        final_extracted_df
        .groupby('Random ID', as_index=False)
        .agg(agg_rules)
    )

    # =========================
    # 5. MAPPINGS
    # =========================
    mapping = [
        ('Radio_Ascites', 'Radio_Ascites_presence', 'Radio_Ascites_date',
         'Ascites (Y/N)', 'First Ascites Date'),

        ('Variceal_bleed', 'Variceal_bleed_presence', 'Variceal_bleed_date',
         'Variceal bleed (Y/N)', 'First Variceal bleed Date'),
        
        ("Portal_Vein_Thrombosis", 'Portal_Vein_Thrombosis_presence', 'Portal_Vein_Thrombosis_date',
         'PVT (Y/N)', 'First PVT Date'),

        ('HE', 'HE_presence', 'HE_date',
         'HE (Y/N)', 'First HE Date'),

        ('SBP', 'Spontaneous_Bacterial_Peritonitis_presence',
         'Spontaneous_Bacterial_Peritonitis_date',
         'SBP (Y/N)', 'First SBP Date'),

        ('HCC', 'HCC_presence', 'HCC_date',
         'HCC (Y/N)', 'First HCC Date'),

        ('TIPS', 'TIPS_presence', 'TIPS_date',
         'TIPS (Y/N)', 'First TIPS Date'),

        ('Clinical_Ascites', 'Clinical_Ascites_presence',
         'Clinical_Ascites_date',
         'Ascites (Y/N)', 'First Ascites Date'),
        
        ('Any_Ascites', 'Any_Ascites_presence', 'Any_Ascites_date',
         'Ascites (Y/N)', 'First Ascites Date'),
        
        ('Any_HCC', 'Any_HCC_presence', 'Any_HCC_date',
         'HCC (Y/N)', 'First HCC Date'),
        
        ('Radio_HCC', 'Radio_HCC_presence', 'Radio_HCC_date',
         'HCC (Y/N)', 'First HCC Date')

        
    ]

    presence_results = []
    date_results = []

    # =========================
    # 6. PER-CONDITION ANALYSIS
    # =========================
    for display_name, pred_p, pred_d, truth_p, truth_d in mapping:

        if (
            pred_p not in final_merged_df.columns or
            truth_p not in df_truth.columns
        ):
            continue

        merged = pd.merge(
            final_merged_df[['Random ID', pred_p, pred_d]],
            df_truth[['Random ID', truth_p, truth_d]],
            on='Random ID',
            how='inner'
        )

        # -------------------------
        # PRESENCE NORMALIZATION
        # -------------------------
        merged[pred_p] = (
            pd.to_numeric(merged[pred_p], errors='coerce')
            .fillna(0)
            .astype(int)
        )
        merged[truth_p] = (
            pd.to_numeric(merged[truth_p], errors='coerce')
            .fillna(0)
            .astype(int)
        )

        # =========================
        # 6A. PRESENCE METRICS
        # =========================
        y_true = merged[truth_p]
        y_pred = merged[pred_p]

        tn, fp, fn, tp = confusion_matrix(
            y_true, y_pred, labels=[0, 1]
        ).ravel()

        presence_results.append({
            'Condition': display_name,
            'TP': tp,
            'FP': fp,
            'TN': tn,
            'FN': fn,
            'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred, zero_division=0),
            'Recall': recall_score(y_true, y_pred, zero_division=0),
            'F1-Score': f1_score(y_true, y_pred, zero_division=0),
        })

        # =========================
        # 6B. DATE METRICS (TP ONLY)
        # =========================
        tp_subset = merged[
            (merged[pred_p] == 1) &
            (merged[truth_p] == 1)
        ].copy()

        if tp_subset.empty:
            continue

        y_true_date = pd.to_datetime(
            tp_subset[truth_d],
            dayfirst=True,
            errors='coerce'
        )
        y_pred_date = pd.to_datetime(
            tp_subset[pred_d],
            errors='coerce'
        )

        valid_mask = y_true_date.notna() & y_pred_date.notna()
        diff = (y_true_date[valid_mask] - y_pred_date[valid_mask]).dt.days.abs()

        if diff.empty:
            continue

        date_results.append({
            'Condition': display_name,
            'True_Positives': len(tp_subset),
            'Dates_Compared': len(diff),
            'Exact_Match': f"{(diff == 0).sum()} ({round((diff == 0).mean()*100, 1)}%)",
            '±3 Days': f"{(diff <= 3).sum()} ({round((diff <= 3).mean()*100, 1)}%)",
            '±7 Days': f"{(diff <= 7).sum()} ({round((diff <= 7).mean()*100, 1)}%)",
            '±30 Days': f"{(diff <= 30).sum()} ({round((diff <= 30).mean()*100, 1)}%)",
            'Median_Abs_Error_Days': int(diff.median())
        })

    return (
        pd.DataFrame(presence_results),
        pd.DataFrame(date_results)
    )


In [13]:
presence_metrics, tp_date_metrics = calculate_medical_metrics_with_dates(
    extracted_csv_paths=[
        "1k SBP 6.5.csv",
        "1k TIPS 6.5.csv",
        "1k VARICEAL 25.5.csv",
        "1k HE 6.5.csv",
        "1k PVT 7.5.csv",
        "1k HCC 25.5.csv",
        "1k CASCITES 6.5.csv",
        "1k RASCITES 6.5.csv",
        "1k AASCITES.csv",
        "1k RHCC 20.5.csv",
        "1k ANY_HCC.csv"
    ]
)

print("=== PRESENCE METRICS ===")
print(presence_metrics)

print("\n=== TP DATE METRICS ===")
print(tp_date_metrics)

=== PRESENCE METRICS ===
                 Condition   TP   FP    TN   FN  Accuracy  Precision  \
0            Radio_Ascites  278  120   569  133  0.770000   0.698492   
1           Variceal_bleed  161  132   784   23  0.859091   0.549488   
2   Portal_Vein_Thrombosis  118   71   903    8  0.928182   0.624339   
3                       HE  129  101   850   20  0.890000   0.560870   
4                      SBP   53   99   945    3  0.907273   0.348684   
5                      HCC  231   82   768   19  0.908182   0.738019   
6                     TIPS   45   18  1034    3  0.980909   0.714286   
7         Clinical_Ascites  367  128   561   44  0.843636   0.741414   
8              Any_Ascites  388  186   503   23  0.810000   0.675958   
9                  Any_HCC  227   80   770   23  0.906364   0.739414   
10               Radio_HCC  222   77   773   28  0.904545   0.742475   

      Recall  F1-Score  
0   0.676399  0.687268  
1   0.875000  0.675052  
2   0.936508  0.749206  
3   0.8657

In [12]:
presence_metrics, tp_date_metrics = calculate_medical_metrics_with_dates(
    extracted_csv_paths=[      
        "1k VARICEAL 26.5.csv",
        "1k HCC 25.5.csv"
    ]
)

print("=== PRESENCE METRICS ===")
print(presence_metrics)

print("\n=== TP DATE METRICS ===")
print(tp_date_metrics)

=== PRESENCE METRICS ===
        Condition   TP   FP   TN  FN  Accuracy  Precision   Recall  F1-Score
0  Variceal_bleed  159  129  787  25  0.860000   0.552083  0.86413  0.673729
1             HCC  231   82  768  19  0.908182   0.738019  0.92400  0.820604

=== TP DATE METRICS ===
        Condition  True_Positives  Dates_Compared Exact_Match     ±3 Days  \
0  Variceal_bleed             159             158  51 (32.3%)  75 (47.5%)   
1             HCC             231             227    1 (0.4%)   20 (8.8%)   

      ±7 Days    ±30 Days  Median_Abs_Error_Days  
0  76 (48.1%)  86 (54.4%)                     20  
1  25 (11.0%)  43 (18.9%)                    121  


### SBP Lab Evaluation ( I didn't end up using this )
Auto evaluates SBP based on PMN count, formula is as follows:\
Main formula `(WBC *1000)* (NEUTROPHILIS / 100)` \
unit for WBC : X10(9)/L\
unit for Neutrophilis : %

In [ ]:
import pandas as pd
import numpy as np

def calculate_sbp_status(df_labs, df_patients, time_tolerance='2D'):
    print("--- DEBUG START ---")
    print(f"Initial df_labs rows: {len(df_labs)}")
    
    # 1. Clean data
    df_labs = df_labs.copy()
    df_labs['Reported Date'] = pd.to_datetime(df_labs['Reported Date'], errors='coerce')
    df_labs['Result Value'] = pd.to_numeric(df_labs['Result Value'], errors='coerce')
    df_labs = df_labs.dropna(subset=['Reported Date', 'Result Value'])
    print(f"Rows after cleaning dates/values: {len(df_labs)}")

    # 2. Split labs
    df_wbc = df_labs[df_labs['Lab Resulted Order Test Description'].str.contains('FLUID DIFF CT|FLUID DIFFERENTIAL COUNT', case=False, na=False)].copy()
    df_neutro = df_labs[df_labs['Lab Resulted Order Test Description'].str.contains('POLYMORPHONUCLEAR LEUCOCYTES', case=False, na=False)].copy()
    print(f"WBC rows found: {len(df_wbc)}")
    print(f"Neutrophil rows found: {len(df_neutro)}")

    if len(df_wbc) == 0 or len(df_neutro) == 0:
        print("!!! ALERT: 0 rows found for WBC or Neutrophils. Check your column strings!")
        return None

    # 3. Sort
    df_wbc = df_wbc.sort_values('Reported Date')
    df_neutro = df_neutro.sort_values('Reported Date')

    # 4. Merge nearest tests
    df_paired = pd.merge_asof(
        df_wbc, 
        df_neutro, 
        on='Reported Date', 
        by='Random ID', 
        direction='nearest',
        tolerance=pd.Timedelta(time_tolerance),
        suffixes=('_wbc', '_neutro')
    )
    
    # Check how many successfully paired
    df_paired = df_paired.dropna(subset=['Result Value_wbc', 'Result Value_neutro'])
    print(f"Successfully paired tests (within {time_tolerance}): {len(df_paired)}")

    # 5. Calculate PMN
    df_paired['Calculated_PMN'] = df_paired['Result Value_wbc'] * (df_paired['Result Value_neutro'] / 100)
    if not df_paired.empty:
        print(f"Max PMN value found: {df_paired['Calculated_PMN'].max()}")

    # 6. Positive cases
    df_positive = df_paired[df_paired['Calculated_PMN'] > 250].copy()
    print(f"Positive SBP cases (> 250): {len(df_positive)}")

    # 7. Earliest dates
    earliest_dates = df_positive.groupby('Random ID')['Reported Date'].min().reset_index()
    earliest_dates.rename(columns={'Reported Date': 'Date'}, inplace=True)
    earliest_dates['SBP (Y/N)'] = 1

    # 8. Merge back to Master
    df_unique_patients = df_patients[['Random ID']].drop_duplicates()
    final_df = pd.merge(df_unique_patients, earliest_dates, on='Random ID', how='left')
    final_df['SBP (Y/N)'] = final_df['SBP (Y/N)'].fillna(0).astype(int)
    
    print(f"Final Count of SBP Patients: {final_df['SBP (Y/N)'].sum()}")
    print("--- DEBUG END ---")
    return final_df

In [ ]:
df_patients=(df_truth.copy()[['Random ID']]).dropna(subset=['Random ID'])
df_patients
# Run the function
df_sbp_results = calculate_sbp_status(df_labs, df_patients)
pd.DataFrame(df_sbp_results).to_excel('1K SBP_LAB 08.04.xlsx', index=False)
df_sbp_results

In [ ]:
print(calculate_medical_metrics(df_truth, df_sbp_results))